# tensor-unbind composite — cx15: unbind components then re-stack along a new axis (stack vs cat)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `tensor-unbind`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-unbind"
DD_ATOM_IDS = ["tensor-unbind", "stack-vs-cat"]
DD_SUBTOPICS = ["Numpy: Indexing and selection", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A frequent ARENA pattern is the round-trip: take a packed tensor, *unbind* it along one axis to apply per-component transformations, then *stack* the results back into a new packed tensor. The composition exercises the **stack vs cat** distinction sharply: `stack` inserts a NEW axis (grows ndim by 1), while `cat` glues along an EXISTING axis (preserves ndim).

Use case: given `rays: (NR, 2, 3)`, you want to scale the direction component (axis 1, index 1) by `dir_scale` but leave origin unchanged, then reassemble. The natural sequence is
  1. `O, D = t.unbind(rays, dim=1)` — both `(NR, 3)`.
  2. `D_scaled = D * dir_scale`.
  3. `t.stack([O, D_scaled], dim=1)` — reinserts the size-2 axis at position 1, recovering `(NR, 2, 3)`.

Using `t.cat` here would be a bug: `cat([O, D_scaled], dim=1)` produces `(NR, 6)` (concatenated along the 3-axis), losing the structural split.

### Composite Exercise — unbind components then re-stack along a new axis (stack vs cat)

**Atoms exercised together**: `tensor-unbind`, `stack-vs-cat`

Implement `cx15_scale_direction(rays, dir_scale)` that takes `rays: (NR, 2, 3)` (axis 1 = [origin, direction]) and a scalar `dir_scale`, scales each ray's direction vector by `dir_scale` (leaves origins unchanged), and returns the rebuilt `(NR, 2, 3)` tensor.

1. **Unbind** along axis 1: `O, D = t.unbind(rays, dim=1)`.
2. Compute `D_scaled = D * dir_scale`.
3. **Stack** along a NEW axis at position 1: `t.stack([O, D_scaled], dim=1)`. The result shape must equal `rays.shape` exactly — `(NR, 2, 3)`.

Common bug: `t.cat([O, D_scaled], dim=1)` gives `(NR, 6)` — wrong shape, wrong structure. The test catches that explicitly.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx15_scale_direction(rays, dir_scale):
    raise NotImplementedError

def _test_cx15():
    # atom-coverage: enforce the solution uses t.unbind (slicing rays[:,0] would otherwise pass)
    import inspect as _inspect
    _src = _inspect.getsource(cx15_scale_direction)
    assert '.unbind(' in _src, 'solution must use t.unbind (not slicing)'
    # Case A: hand-built rays, scale by 3.
    NR = 4
    O = t.stack([t.tensor([float(i), 0.0, 0.0]) for i in range(NR)])
    D = t.stack([t.tensor([0.0, 1.0, 0.0]) for _ in range(NR)])
    rays = t.stack([O, D], dim=1)
    assert tuple(rays.shape) == (NR, 2, 3)

    out = cx15_scale_direction(rays, 3.0)
    assert tuple(out.shape) == (NR, 2, 3), (
        f'shape changed: {tuple(out.shape)}. '
        'Did you use cat instead of stack? cat would give (NR, 6).'
    )
    # Origins unchanged.
    assert t.allclose(out[:, 0, :], O), 'origins should be unchanged'
    # Directions scaled by 3.
    assert t.allclose(out[:, 1, :], D * 3.0), 'directions should be 3x'

    # Case B: dir_scale = 0 — directions zeroed, origins intact.
    out0 = cx15_scale_direction(rays, 0.0)
    assert t.allclose(out0[:, 0, :], O)
    assert t.allclose(out0[:, 1, :], t.zeros_like(D))

    # Case C: random rays, negative scale.
    rays2 = t.randn(7, 2, 3)
    out2 = cx15_scale_direction(rays2, -1.0)
    assert tuple(out2.shape) == (7, 2, 3)
    assert t.allclose(out2[:, 0, :], rays2[:, 0, :])
    assert t.allclose(out2[:, 1, :], -rays2[:, 1, :])

    # Case D: stress — explicit cross-check via manual axis index.
    expected = rays2.clone()
    expected[:, 1, :] = expected[:, 1, :] * 2.5
    out3 = cx15_scale_direction(rays2, 2.5)
    assert t.allclose(out3, expected)
    _dd_passed.add('cx15')

_test_cx15()

<details><summary>Show solution — cx15</summary>

```python
def cx15_scale_direction(rays, dir_scale):
    # Atom A (tensor-unbind): peel apart the (origin, direction) pair along axis 1.
    O, D = t.unbind(rays, dim=1)
    D_scaled = D * dir_scale
    # Atom B (stack-vs-cat): stack INSERTS a new axis of size 2 at position 1.
    # cat would concatenate along an existing axis -> wrong shape.
    return t.stack([O, D_scaled], dim=1)
```

The unbind/stack pair is the structural inverse: `t.stack(t.unbind(x, dim=d), dim=d)` returns a tensor with the same shape AND values as `x`. That property is what makes this round-trip safe for transformations that don't change per-component shape. If you reach for `t.cat` instead, the axis-1 dimension disappears (it's glued into axis 0 or axis 2 depending on dim) — the shape signal alone catches the bug. `stack` introduces a new axis; `cat` merges along an existing one. Memorize that one-liner.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx15'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx15',
        'subtopics': ["Numpy: Indexing and selection", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()